# Example 2 - Get all the companies in Denmark

# Create a panel for all historical companies in Denmark using Danish CVR

This notebook demonstrates how to get all companies in Denmark as of 2025-12-31, active and disolved, and create a joint dataset. 

## The overall steps coded below are:

### 1. Download the all the data using the python scripts in this repo and store them as `parquet` files.

**IMPORTANT**: If you have not downloaded the data you will need to run the `data_extraction` scripts. This step take some time to request the data (e.g. 30 min to 1 hour).

The strip download all Danish companies with their full history at a given founding year. For example: `python data_extraction/src/get_historical_virksomhed_api_call.py --start-year 1991 --end-year 2000` downloads all the companies founded from 1991 to 2000. All years included.

The oldest CVR in Denmark is University of Copenhagen at  1300, but there are no records until 1735 in the API . I recommend starting from companies founded from 1800 onwards the earliest.

Fell free to check the source code and `readme.md` at `data_extraction/`. 


### 2. Get from the `main` dataset: the last company name and founding date

  - `Vrvirksomhed_cvrNummer` (cvr_number)
  - `Vrvirksomhed_virksomhedMetadata_nyesteNavn_navn` (latest_name)
  - `Vrvirksomhed_virksomhedMetadata_stiftelsesDato` (founding_date)

### 3. Get from `livsforloeb` (company lifecycle) dataset: the activity time stamps

  - `cvrNummer` (cvr_number)
  - `gyldigFra` (valid_from)
  -	`gyldigTil` (valid_to)

This will give from which day to which day the company was active. 

It can include `gyldigFra` *after* `31-12-2025` since a company can be founded in 2025 December and signed to be active in 2026 January, for example.

Values `gyldigTil == None` means that the company is active at the time of the data extraction ("No  end validity"). Therefore, every time you run this query you likely have different set of companies, as companies close daily.

Since I wanted to have a fix point (1700-2025):

  - I deleted any company with `gyldigFra` *after* `31-12-2025`. This means that companies created in 2025 but signed to be active in 2026 are not included.
  - I considered "Active" any company closed in 2026. Closing in 2026 means that they were active in 2025. Effectively, this can be done setting `gyldigFra == None` (active) for companies with a closing date *after* `31-12-2025`.

### 4. Get from `virksomhedsform` (business form) dataset: the company type: 

This is relevant for research since there are some reserach questions that need to limit the data to APS companies only, for example. 

  - `kortBeskrivelse` (short_company_legal_name)
  - `langBeskrivelse` (long_company_legal_name)
  - `gyldigFra` (valid_from)
  -	`gyldigTil` (valid_to)

### 5. Merge datasets (no panel)

Merging all the data together.

Notice the `cvrNummer` is non unique for `livsforloeb` and `virksomhedsform` datasets, and therefore for the final dataset. The same company can appear multiple times as:

- Companies re-open at different times using the same CVR number.
- Companies can  move from one city to another (e.g. https://cvrapi.dk/api?search=12397399&country=dk)
- Companies change their legal structure, or shut down for some years (e.g. https://cvrapi.dk/api?search=10000173&country=dk). These companies have different `productionunits` ID (identification similar as CVR) for every time they re-open.


In [82]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

pd.set_option('display.max_columns', None)
load_dotenv()

PROJECT_HOME_PATH = "/Users/pipegalera/dev/GitHub/CVR_data"
RAW_VIRKSOMHED_FOLDER_PATH = os.getenv("RAW_VIRKSOMHED_FOLDER_PATH")
PROC_VIRKSOMHED_FOLDER_PATH = os.getenv("PROC_VIRKSOMHED_FOLDER_PATH")

os.chdir(PROJECT_HOME_PATH)

from utils.translations import COLUMN_TRANSLATIONS, VALUE_TRANSLATIONS

## 1. Download the data using the python scripts

In [53]:
# See "Example 1 -  Download and read data" for step-by-step
#!python data_extraction/src/get_historical_virksomhed_api_call.py --start-year 1800 --end-year 2026

## 2. Get from the `main` dataset the last name and founding date

Fields extracted:
- `cvr_number`
- `latest_name` — from nested `virksomhedMetadata['nyesteNavn']['navn']`
- `founding_date` — from nested `virksomhedMetadata['stiftelsesDato']`

In [54]:
YEARS = range(1800, 2027)
def merge_parquets(table_name, folder_path=RAW_VIRKSOMHED_FOLDER_PATH):
    dfs = []
    for year in range(1800, 2006):
        file_path = f"{folder_path}/{table_name}_{year}.parquet"
        if os.path.exists(file_path):
            dfs.append(pd.read_parquet(file_path))
    df = pd.concat(dfs, ignore_index=True)
    return df

In [55]:
main_raw = merge_parquets("main")

In [56]:
main_raw

,cvrNummer,brancheAnsvarskode,reklamebeskyttet,virksomhedMetadata,samtId,fejlRegistreret,dataAdgang,enhedsNummer,enhedstype,sidstIndlaest,sidstOpdateret,fejlVedIndlaesning,naermesteFremtidigeDato,fejlBeskrivelse,virkningsAktoer,fortroligBeriget
0,36515279,None,False,"{'antalPenheder': 0, 'nyesteAarsbeskaeftigelse...",1,False,1,4004359841,VIRKSOMHED,2026-02-03T01:27:41.214+01:00,2015-02-09T23:00:00.000+01:00,False,None,None,R,NaN
1,66276716,0,False,"{'antalPenheder': 7, 'nyesteAarsbeskaeftigelse...",10,False,0,4001180074,VIRKSOMHED,2026-05-20T09:46:53.562+02:00,2023-01-31T12:59:09.000+01:00,False,None,None,R,NaN
2,29157812,0,True,"{'antalPenheder': 28, 'nyesteAarsbeskaeftigels...",47,False,0,4000333232,VIRKSOMHED,2026-05-27T00:18:52.649+02:00,2023-02-01T08:01:55.000+01:00,False,None,None,R,NaN
3,21016411,None,True,"{'antalPenheder': 1, 'nyesteAarsbeskaeftigelse...",223,False,0,4001060009,VIRKSOMHED,2026-05-31T02:04:57.182+02:00,2026-01-21T09:08:44.000+01:00,True,None,Medlem: Ukendt deltager enhedsnummer for cvr: ...,PO,NaN
4,45657116,None,False,"{'antalPenheder': 1, 'nyesteAarsbeskaeftigelse...",5,False,0,4001131612,VIRKSOMHED,2025-09-27T00:18:28.599+02:00,2023-02-01T10:54:33.000+01:00,False,None,None,R,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
865707,29149682,NaN,False,"{'antalPenheder': 1, 'nyesteAarsbeskaeftigelse...",14,False,0,4001531212,VIRKSOMHED,2026-06-07T15:34:32.665+02:00,2026-02-16T14:21:48.000+01:00,False,NaN,NaN,PO,None
865708,29204497,NaN,True,"{'antalPenheder': 1, 'nyesteAarsbeskaeftigelse...",11,False,0,4001548306,VIRKSOMHED,2026-06-07T15:48:10.677+02:00,2026-02-16T14:21:48.000+01:00,False,NaN,NaN,PO,None
865709,28328753,NaN,False,"{'antalPenheder': 1, 'nyesteAarsbeskaeftigelse...",16,False,0,4001498263,VIRKSOMHED,2026-06-07T15:24:02.120+02:00,2026-03-24T15:39:08.000+01:00,True,NaN,Medlem: Ukendt deltager enhedsnummer for cvr: ...,PO,None
865710,28856334,NaN,False,"{'antalPenheder': 1, 'nyesteAarsbeskaeftigelse...",11,False,0,4001530560,VIRKSOMHED,2026-06-07T15:58:31.031+02:00,2026-03-04T12:11:47.000+01:00,False,NaN,NaN,PO,None


In [57]:
main = pd.DataFrame({
    "cvr_number": main_raw["cvrNummer"],
    "latest_name": main_raw["virksomhedMetadata"].apply(
        lambda x: x["nyesteNavn"]["navn"] if isinstance(x, dict) and x.get("nyesteNavn") else None
    ),
    "founding_date": main_raw["virksomhedMetadata"].apply(
        lambda x: x.get("stiftelsesDato") if isinstance(x, dict) else None
    ),
})

In [58]:
main

,cvr_number,latest_name,founding_date
0,36515279,SPAREKASSEN FOR GREVSKABET HOLSTEINBORG OG OMEGN,1810-11-02
1,66276716,SPAREKASSEN LØGUMKLOSTER,1821-01-12
2,29157812,SPAREKASSEN SJÆLLAND,1825-12-23
3,21016411,Fonden Det Københavnske Asylselskab ERF,1835-12-01
4,45657116,RK FORSIKRING G/S,1838-03-13
...,...,...,...
865707,29149682,SPEJLBJERG INVEST ApS,2005-10-26
865708,29204497,ALEKS OLESEN HOLDING ApS,2005-12-13
865709,28328753,ADMI-CONSULT ApS,2005-01-18
865710,28856334,ML OLESEN ApS,2005-06-23


## 3. Get from `livsforloeb` (lifecycle) dataset the activity timestamps

Fields:
- `cvr_number`
- `gyldigFra` (valid_from) — from `periode_gyldigFra`
- `gyldigTil` (valid_to) — from `periode_gyldigTil`

`gyldigTil == None` means the company is active at time of extraction.

Temporal cutoff set to 31-12-2025:
- Companies with `gyldigFra` after 2025-12-31 are dropped.
- Companies with `gyldigTil` after 2025-12-31 are treated as active (set to `None`).

In [59]:
lifecycle = merge_parquets("livsforloeb")[["cvrNummer", "periode_gyldigFra", "periode_gyldigTil"]]
lifecycle = lifecycle.rename(columns={
    "cvrNummer": "cvr_number",
    "periode_gyldigFra": "gyldigFra",
    "periode_gyldigTil": "gyldigTil",
})

In [60]:
lifecycle

,cvr_number,gyldigFra,gyldigTil
0,36515279,1810-11-02,1989-01-01
1,66276716,1821-01-12,2009-03-12
2,29157812,1825-12-23,2015-11-24
3,21016411,1835-12-01,None
4,45657116,1838-03-13,2007-08-09
...,...,...,...
970597,29149682,2005-10-26,NaN
970598,29204497,2005-12-13,NaN
970599,28328753,2005-01-18,NaN
970600,28856334,2005-06-23,NaN


In [61]:
# Sample of companies that closed operations in 1st Jan 2026
lifecycle[lifecycle["gyldigTil"] == "2026-01-01"].sample(5)

,cvr_number,gyldigFra,gyldigTil
874653,27844006,2004-09-01,2026-01-01
871910,27855741,2026-01-01,2026-01-01
177405,12397399,2022-04-01,2026-01-01
532055,20639903,1998-03-01,2026-01-01
755969,26428912,2002-01-23,2026-01-01


In [62]:
# Sample of companies that are active as of current date (e.g. early June 2026)
# You can check the CVRs manually using this website: https://cvrapi.dk/
lifecycle[lifecycle["gyldigTil"].isna()].sample(5)

,cvr_number,gyldigFra,gyldigTil
595570,22028898,1999-10-01,NaN
773486,26722535,2002-06-27,NaN
353070,15790342,1991-12-30,NaN
239550,10033098,1986-01-01,NaN
497105,19845583,1997-01-01,NaN


In [63]:
def set_temporal_cutoff(data, cutoff_date):
    df = data.copy()
    df["gyldigTil"] = pd.to_datetime(df["gyldigTil"], errors='coerce')
    df["gyldigTil"] = df["gyldigTil"].where(df["gyldigTil"] <= pd.Timestamp(cutoff_date), None)
    return df

lifecycle = set_temporal_cutoff(lifecycle, cutoff_date="2025-12-31")

In [64]:
lifecycle

,cvr_number,gyldigFra,gyldigTil
0,36515279,1810-11-02,1989-01-01
1,66276716,1821-01-12,2009-03-12
2,29157812,1825-12-23,2015-11-24
3,21016411,1835-12-01,NaT
4,45657116,1838-03-13,2007-08-09
...,...,...,...
970597,29149682,2005-10-26,NaT
970598,29204497,2005-12-13,NaT
970599,28328753,2005-01-18,NaT
970600,28856334,2005-06-23,NaT


In [65]:
# After setting closed companies in 2026 as active (None) for 2025.
test_cvrs = [12354134, 12397399, 44573350, 17743406, 25625714,
             11088988, 86691450, 28815212, 12992742, 95589855]
lifecycle[lifecycle["cvr_number"].isin(test_cvrs)].sort_values(["cvr_number", "gyldigFra"])
# As an example, 12992742 closed multiple times, but since the last one is in 2026 counts as active (None)

,cvr_number,gyldigFra,gyldigTil
234824,11088988,1986-03-01,NaT
275827,12354134,1988-01-01,NaT
177404,12397399,1983-04-19,2022-03-12
177405,12397399,2022-04-01,NaT
73172,12992742,1972-09-15,2008-06-30
73173,12992742,2023-06-01,2023-08-30
73174,12992742,2025-11-01,NaT
412229,17743406,1994-05-03,2004-12-31
412230,17743406,2026-01-01,NaT
638594,25625714,2000-09-18,NaT


In [66]:
lifecycle[lifecycle["gyldigFra"] > "2025-12-31"]

,cvr_number,gyldigFra,gyldigTil
5559,11677398,2026-05-07,NaT
23134,22245228,2026-05-07,NaT
27737,35657916,2026-04-01,NaT
46273,13194416,2026-01-01,NaT
54213,28798814,2026-05-01,NaT
...,...,...,...
968948,28456905,2026-03-03,NaT
969806,28537166,2026-04-16,NaT
969894,29097887,2026-03-05,NaT
970215,28997019,2026-05-11,NaT


In [67]:
# Drop companies whose activity only starts after the cutoff
lifecycle = lifecycle[lifecycle["gyldigFra"] <= "2025-12-31"]

In [69]:
# Should be empty
lifecycle[lifecycle["gyldigFra"] > "2025-12-31"]

,cvr_number,gyldigFra,gyldigTil


## 4. Get from `virksomhedsform` the company legal status fields

Fields:

  - `cvr_number`
  - `kortBeskrivelse` (short_company_legal_name)
  - `langBeskrivelse` (long_company_legal_name)
  - `gyldigFra` (valid_from)
  -	`gyldigTil` (valid_to)

In [70]:
legal_form = merge_parquets("virksomhedsform")[["cvrNummer", "kortBeskrivelse", "langBeskrivelse", "periode_gyldigFra", "periode_gyldigTil"]]
legal_form = (legal_form
    .rename(columns={
        "cvrNummer": "cvr_number",
        "kortBeskrivelse": "short_company_legal_name",
        "langBeskrivelse": "long_company_legal_name",
        "periode_gyldigFra": "legal_form_from",
        "periode_gyldigTil": "legal_form_to",
    })
    .replace(VALUE_TRANSLATIONS)
)

In [71]:
legal_form

,cvr_number,short_company_legal_name,long_company_legal_name,legal_form_from,legal_form_to
0,36515279,UOP,Uoplyst virksomhedsform,1810-11-02,1989-01-01
1,66276716,UOP,Uoplyst virksomhedsform,1821-01-12,2009-01-20
2,66276716,ØVR,other_business_forms,2009-01-21,2009-03-12
3,29157812,UOP,Uoplyst virksomhedsform,1825-12-23,2009-01-20
4,29157812,FIV,special_financial_enterprise,2009-01-21,2015-11-24
...,...,...,...,...,...
961845,29149682,APS,private_limited_company,2005-10-26,NaN
961846,29204497,APS,private_limited_company,2005-12-13,NaN
961847,28328753,APS,private_limited_company,2005-01-18,NaN
961848,28856334,APS,private_limited_company,2005-06-23,NaN


## 5. Merge datasets (no panel)

Merging all the data together (main + lifecycle + legal_form).

Notice the `cvrNummer` is non unique for `livsforloeb` and `virksomhedsform` datasets, and therefore the final dataset will have more rows than `main`. 

Why? The same company can appear multiple times as:

- Companies re-open at different times using the same CVR number.
- Companies can  move from one city to another (e.g. https://cvrapi.dk/api?search=12397399&country=dk)
- Companies change their legal structure, or shut down for some years (e.g. https://cvrapi.dk/api?search=10000173&country=dk). These companies have different `productionunits` ID (identification similar as CVR) for every time they re-open.

In [81]:
lifecycle_renamed = lifecycle.rename(columns={"gyldigFra": "company_from", "gyldigTil": "company_to"})

df = (main
          .merge(lifecycle_renamed, on="cvr_number", how="outer")
          .merge(legal_form, on="cvr_number", how="outer"))

print("Rows:")
print("______________________")
print(f"Main:       {len(main):,}")
print(f"Lifecycle:  {len(lifecycle_renamed):,}")
print(f"Legal form: {len(legal_form):,}")
print(f"Merged:     {len(df):,}")

Rows:
______________________
Main:       865,712
Lifecycle:  969,290
Legal form: 961,850
Merged:     1,167,793


In [73]:
df # All companies in Denmark as of 2025-12-31

,cvr_number,latest_name,founding_date,company_from,company_to,short_company_legal_name,long_company_legal_name,legal_form_from,legal_form_to
0,10000009,YELLOW ApS,1999-10-12,1999-10-12,2001-12-11,APS,private_limited_company,1999-10-12,2001-12-11
1,10000025,WATERFRONT CONNECTION ApS,1999-10-13,1999-10-13,NaT,APS,private_limited_company,1999-10-13,NaN
2,10000068,"STUDENTCONSULTING, FILIAL AF SVERIGES STU...",1999-10-18,1999-10-18,2001-12-20,FAS,branch_of_foreign_public_limited_company,1999-10-18,2001-12-20
3,10000106,TRANBJERG TAGDÆKNING V/JOHN HARTM...,1985-07-11,1985-07-11,2007-06-30,ENK,sole_proprietorship,1985-07-11,2007-06-30
4,10000122,DIGITAL CENTER FYN ApS,1999-10-14,1999-10-14,2002-08-15,APS,private_limited_company,1999-10-14,2002-08-15
...,...,...,...,...,...,...,...,...,...
1167788,99995653,Skovly FerieCenter,1986-07-01,1986-07-01,2004-01-01,ENK,sole_proprietorship,2019-05-28,NaN
1167789,99995653,Skovly FerieCenter,1986-07-01,2019-05-28,NaT,ENK,sole_proprietorship,1986-07-01,2004-01-01
1167790,99995653,Skovly FerieCenter,1986-07-01,2019-05-28,NaT,ENK,sole_proprietorship,2019-05-28,NaN
1167791,99997451,SCHÆFERKLUBBEN KREDS 52,1986-06-10,1986-06-10,NaT,FOR,association,1986-06-10,NaN
